# F2 painterly composition v0.3

A: preserved v0.2 E. B: absolute bottom boundary + selective positive terrain + softened far-foot omission. C: B plus a few larger representative canopies, with original small/middle/far marks retained.

Install `python -m pip install -e ".[research,dev]"` once from the repository root and choose that environment in VSCode/Jupyter. Keep original TIFFs outside Git and set `DATASCAPE_DATA_DIR` to their folder. Camera, projection, source heights and v0.1/v0.2 results remain unchanged.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
from IPython.display import display, Image
from datascape.rendering.composition_refinement import run
from datascape.composition.refinement import boundary_retention, apply_boundary

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
CONFIG = ROOT / "configs/scenes/seoul_f2_painterly_composition_v03.json"
assert CONFIG.is_file()
DATA = Path(os.environ.get("DATASCAPE_DATA_DIR", ROOT / "data/raw"))
MASTER = DATA / "SeoulNorth_v02_MASTER_30m.tif"
VEG = DATA / "SeoulNorth_v02_VEG_10m.tif"
assert MASTER.is_file() and VEG.is_file()
OUTPUT = ROOT / "outputs/f2_painterly_composition_v03"


In [ ]:
result = run(MASTER, VEG, CONFIG, OUTPUT, repo_root=ROOT)
display(Image(filename=str(OUTPUT / "ABC_comparison.png")))
display(Image(filename=str(OUTPUT / "C_physical_pair.png")))

## Boundary priority and positive forms

The hard guard and universal collar are separate from the irregular 5–15% transition. Every layer receives the boundary gate; terrain/root/depth exceptions cannot override it. Raster coverage uses pixel lower edges, so the final row includes the exact 900 mm boundary even if guard width is zero.

Positive forms are selected globally from DEM relative height, gentle slope and local relief. There are no prescribed X bins. Small-root supports are a separately recorded legibility preference at existing measured root sites, not evidence of rocks or trees.

In [ ]:
for name in ["fields.png", "boundary_diagnostics.png", "near_comparison.png", "far_comparison.png"]:
    display(Image(filename=str(OUTPUT / name)))
metrics = result["record"]["metrics"]
print(json.dumps(metrics, indent=2))

In [ ]:
r = result["reserves"]
for name in ["terrain_ink", "vegetation_B", "vegetation_C", "ink_B", "ink_C"]:
    assert np.all(result[name][-1] == 0), name
assert float(boundary_retention(900, **result["record"]["config"]["boundary"])) == 0
# The same API can gate future color channels without a color engine.
channels = apply_boundary(np.ones((3, 4)), np.array([860., 885., 900.]),
                          **result["record"]["config"]["boundary"])
assert np.all(channels[-1] == 0)
assert metrics["old_near_field_identical"] and metrics["old_middle_field_identical"]
assert metrics["F2_baseline_pixels_identical"] and metrics["v02_E_pixels_identical"]
assert metrics["previous_artifacts_unchanged"]
assert result["record"]["inputs"]["master"]["unchanged_after"]
assert result["record"]["inputs"]["vegetation"]["unchanged_after"]
if (OUTPUT / "representative_depth.png").is_file():
    display(Image(filename=str(OUTPUT / "representative_depth.png")))
print("Verified exact boundary omission, immutable inputs/baselines and depth-aware scale comparison.")

## Handoff to vector studies

This is the last composition-mask comparison. Keep the selected F2 scene and these omission fields fixed when comparing Taebaek horizontal curves, terrain structure lines and direction-field strokes.

Later vector consumers should split/clip segments at the configured hard bottom limit, then use the same continuous retention for density or layer inclusion. Future color/ink contributions use the final boundary gate as well. No vector stroke engine, final species treatment, color field or G-code is implemented in this notebook.

For a controlled composition variation, copy the JSON and write to a new directory. Representative height/count, root spacing/seam clearance, form footprint, irregular transition and boundary guard/collar are independent recorded parameters.